# Exploiter le guide des recettes

## Set variables

In [85]:
from dotenv import dotenv_values
config = dotenv_values("../../.env")

llm_model = config.get('ONLINE_LLM_MODEL')
api_key = config.get('ONLINE_LLM_API_KEY')
qdrant_url = config.get('QDRANT_URL')
qdrant_api_key = config.get('QDRANT_API_KEY')

## Configuration du llm sur dspy

In [86]:
import dspy
from dspy_qdrant import QdrantRM
from qdrant_client import QdrantClient
from langchain_huggingface import HuggingFaceEmbeddings

# Ton vectorizer (embedding) francophone
hf_embed = HuggingFaceEmbeddings(
    model_name="manu/bge-fr-en",  # modèle francophone + multilingue
    encode_kwargs={"normalize_embeddings": True}
)

def vectorizer(text_or_texts):
    if isinstance(text_or_texts, list):
        return hf_embed.embed_documents(text_or_texts)
    else:
        return hf_embed.embed_query(text_or_texts)


lm = dspy.LM(llm_model, api_key=api_key)
# Uncomment for local api call
#lm = dspy.LM(llm_model, api_base=api_base, track_usage=True, temperature=1.5, max_tokens=1024)

collection_name = "livre-recette"
client = QdrantClient(qdrant_url, api_key=qdrant_api_key)
rm = QdrantRM(
    qdrant_collection_name=collection_name, 
    qdrant_client=client,
    vector_name="recette",
    document_field="text",
    vectorizer=vectorizer,
    k=20
)


dspy.configure_cache(
    enable_disk_cache=False,
    enable_memory_cache=False,
)
dspy.configure(lm=lm, rm=rm)

## Créer le RAG

In [87]:
def rag(query: str, k: int) -> list[str]:
    retriever = dspy.Retrieve(k=k)
    revelant_passages = retriever(query).passages
    # results = [x['text'] for x in results]

    # for result in results:
    #     title, text = result.split(" | ", 1)
    print(f"Revelant passages: {revelant_passages}")
    assistant = dspy.ChainOfThought("context, question -> response")
    return assistant(context=revelant_passages, question=query)

## Exécuter le RAG

In [88]:
from rich import print
results = rag(query="Comment cuisiner un nachos garnis ?", k=2)

print("----------")
print(results.response)
print("----------")
print(lm.history)
print("----------")
dspy.inspect_history()
print("----------")

Revelant passages: ["title=LIVRE DE RECETTES POUR ETUDIANTS PAUVRES > NACHOS GARNIS entrees (A PARTAGER) > 
PREPARATION | text=1. Préchauffer le four à 400 ° F\n2. Sur une plaque de cuisson tapissée de papier-parchemin, 
étendre les croustilles de maïs.\n3. Ajouter les olives, les tomates fraiches et le poivron. Si désiré, le piment 
jalapeño et la salsa.\n4. Garnir de cheddar râpé.\n5. Cuire pendant environ 10 minutes ou jusqu'à ce que le fromage
ait fondu et qu'il soit légèrement doré.", "title=LIVRE DE RECETTES POUR ETUDIANTS PAUVRES > NACHOS GARNIS entrees 
(A PARTAGER) > INGREDIENTS | text=- 4 tasses de croustilles de maïs\n- 3 c. à table (45 ml) d'olives noires coupées
en tranches\n- 3 c. à table (45 ml) de tomates fraîches, épépinées et hachées (ou tomates en conserve)\n- 1 
tasse(250 ml) de cheddar râpé\n- 1/4 tasse (60 ml) de salsa (facultatif)\n- 1 c. à table (15 ml) de piment jalapeño
haché (facultatif)"]

----------

Pour cuisiner un nachos garnis, commencez par préchauffer votre four à 400 °F. Étalez les croustilles de maïs sur 
une plaque recouverte de papier-parchemin. Ajoutez les olives, les tomates fraîches coupées en morceaux, le 
poivron, et si vous souhaitez, le piment jalapeño et la salsa. Saupoudrez ensuite de cheddar râpé. Placez la plaque
au four et faites cuire pendant environ 10 minutes, ou jusqu'à ce que le fromage ait fondu et soit légèrement doré.
Servez chaud et dégustez.

----------

[
    {
        'prompt': None,
        'messages': [
            {
                'role': 'system',
                'content': 'Your input fields are:\n1. `context` (str): \n2. `question` (str):\nYour output fields 
are:\n1. `reasoning` (str): \n2. `response` (str):\nAll interactions will be structured in the following way, with 
the appropriate values filled in.\n\n[[ ## context ## ]]\n{context}\n\n[[ ## question ## ]]\n{question}\n\n[[ ## 
reasoning ## ]]\n{reasoning}\n\n[[ ## response ## ]]\n{response}\n\n[[ ## completed ## ]]\nIn adhering to this 
structure, your objective is: \n        Given the fields `context`, `question`, produce the fields `response`.'
            },
            {
                'role': 'user',
                'content': "[[ ## context ## ]]\n[1] «««\n    title=LIVRE DE RECETTES POUR ETUDIANTS PAUVRES > 
NACHOS GARNIS entrees (A PARTAGER) > PREPARATION | text=1. Préchauffer le four à 400 ° F\n    2. Sur une plaque de 
cuisson tapissée de papier-parchemin, étendre les croustilles de maïs.\n    3. Ajouter les olives, les tomates 
fraiches et le poivron. Si désiré, le piment jalapeño et la salsa.\n    4. Garnir de cheddar râpé.\n    5. Cuire 
pendant environ 10 minutes ou jusqu'à ce que le fromage ait fondu et qu'il soit légèrement doré.\n»»»\n[2] «««\n   
title=LIVRE DE RECETTES POUR ETUDIANTS PAUVRES > NACHOS GARNIS entrees (A PARTAGER) > INGREDIENTS | text=- 4 tasses
de croustilles de maïs\n    - 3 c. à table (45 ml) d'olives noires coupées en tranches\n    - 3 c. à table (45 ml) 
de tomates fraîches, épépinées et hachées (ou tomates en conserve)\n    - 1 tasse(250 ml) de cheddar râpé\n    - 
1/4 tasse (60 ml) de salsa (facultatif)\n    - 1 c. à table (15 ml) de piment jalapeño haché 
(facultatif)\n»»»\n\n[[ ## question ## ]]\nComment cuisiner un nachos garnis ?\n\nRespond with the corresponding 
output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## response ## ]]`, and then ending with 
the marker for `[[ ## completed ## ]]`."
            }
        ],
        'kwargs': {},
        'response': ModelResponse(
            id='chatcmpl-CzRKC1e7m3xUzH3UF4XJXlXOxZpTH',
            created=1768759080,
            model='gpt-4.1-nano-2025-04-14',
            object='chat.completion',
            system_fingerprint='fp_f0bc439dc3',
            choices=[
                Choices(
                    finish_reason='stop',
                    index=0,
                    message=Message(
                        content="[[ ## reasoning ## ]]\nPour cuisiner un nachos garnis, il faut d'abord préchauffer
le four à 400 °F. Ensuite, étaler les croustilles de maïs sur une plaque recouverte de papier-parchemin. Ajouter 
les garnitures comme les olives, les tomates fraîches et le poivron, ainsi que le piment jalapeño et la salsa si 
désiré. Parsemer de fromage cheddar râpé. Enfin, cuire le tout au four pendant environ 10 minutes, jusqu'à ce que 
le fromage fonde et devienne légèrement doré. Ce procédé permet de préparer un nachos chaud et savoureux, idéal 
pour partager.\n\n[[ ## response ## ]]\nPour cuisiner un nachos garnis, commencez par préchauffer votre four à 400 
°F. Étalez les croustilles de maïs sur une plaque recouverte de papier-parchemin. Ajoutez les olives, les tomates 
fraîches coupées en morceaux, le poivron, et si vous souhaitez, le piment jalapeño et la salsa. Saupoudrez ensuite 
de cheddar râpé. Placez la plaque au four et faites cuire pendant environ 10 minutes, ou jusqu'à ce que le fromage 
ait fondu et soit légèrement doré. Servez chaud et dégustez.\n\n[[ ## completed ## ]]",
                        role='assistant',
                        tool_calls=None,
                        function_call=None,
                        provider_specific_fields={'refusal': None},
                        annotations=[]
                    ),
                    provider_specific_fields={}
                )
            ],
            usage=Usage(
                completion_tokens=272,
     

----------





[2026-01-18T18:58:01.233314]

System message:

Your input fields are:
1. `context` (str): 
2. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `response` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## context ## ]]
{context}

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## response ## ]]
{response}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `context`, `question`, produce the fields `response`.


User message:

[[ ## context ## ]]
[1] «««
    title=LIVRE DE RECETTES POUR ETUDIANTS PAUVRES > NACHOS GARNIS entrees (A PARTAGER) > PREPARATION | text=1. Préchauffer le four à 400 ° F
    2. Sur une plaque de cuisson tapissée de papier-parchemin, étendre les croustilles de maïs.
    3. Ajouter les olives, les tomates fraiches et le poivron. Si désiré, le piment jalapeño et la salsa.
    4. Garnir de cheddar râpé.
    5. Cuire p

----------